In [2]:
!pip install -q transformers datasets accelerate

In [3]:


from __future__ import annotations

import argparse
import ast
import json
import os
import re
import subprocess
import sys
import tempfile
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


DEFAULT_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
DEFAULT_SYSTEM_PROMPT = "You are an expert Python competitive programmer."


@dataclass
class Task:
    task_id: str
    prompt: str
    test: str
    entry_point: str


@dataclass
class TokenTrace:
    position: int
    token_id: int
    token_text: str
    entropy: float
    margin: float
    top1_id: int
    top1_text: str
    top2_id: int
    top2_text: str


@dataclass
class Generation:
    token_ids: list[int]
    code: str
    trace: list[TokenTrace]
    latency_s: float


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model-name", default=DEFAULT_MODEL)
    parser.add_argument("--output-dir", default="/kaggle/working/top2_counterfactual_pilot")
    parser.add_argument("--task-ids", default="", help="Comma-separated HumanEval task IDs.")
    parser.add_argument("--num-tasks", type=int, default=10, help="Used only when --task-ids is empty.")
    parser.add_argument("--max-new-tokens", type=int, default=256)
    parser.add_argument("--candidates-per-task", type=int, default=3)
    parser.add_argument("--controls-per-task", type=int, default=3)
    parser.add_argument("--edge-buffer", type=int, default=5)
    parser.add_argument("--timeout-s", type=int, default=8)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--trust-remote-code", action="store_true")
    parser.add_argument("--overwrite", action="store_true")
    # Jupyter/Kaggle executes a cell with an internal ``-f <kernel.json>``
    # argument. Accept that one argument pair while keeping normal CLI typos
    # visible to the user.
    args, unknown = parser.parse_known_args()
    if unknown:
        if len(unknown) == 2 and unknown[0] == "-f":
            return args
        parser.error(f"unrecognized arguments: {' '.join(unknown)}")
    return args


def append_jsonl(path: Path, record: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def load_tasks(task_ids: str, num_tasks: int) -> list[Task]:
    dataset = load_dataset("openai_humaneval", split="test")
    requested_ids = {item.strip() for item in task_ids.split(",") if item.strip()}
    tasks = [
        Task(
            task_id=row["task_id"],
            prompt=row["prompt"],
            test=row["test"],
            entry_point=row["entry_point"],
        )
        for row in dataset
        if not requested_ids or row["task_id"] in requested_ids
    ]
    if requested_ids:
        missing = requested_ids - {task.task_id for task in tasks}
        if missing:
            raise ValueError(f"Unknown HumanEval task IDs: {sorted(missing)}")
        return tasks
    return tasks[:num_tasks]


def build_prompt(task: Task, tokenizer: Any) -> str:
    user_prompt = (
        "Complete the following Python function.\n"
        "Return only valid Python code. Do not use Markdown. Do not explain.\n\n"
        f"{task.prompt}"
    )
    messages = [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"{DEFAULT_SYSTEM_PROMPT}\n\n{user_prompt}"


def strip_markdown_fences(text: str) -> str:
    text = (text or "").strip()
    blocks = re.findall(r"```(?:python|py)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    blocks = [block.strip() for block in blocks if block.strip()]
    if blocks:
        keywords = ("def ", "import ", "from ", "class ", "return ", "assert ")
        return max(blocks, key=lambda block: sum(key in block for key in keywords) * 10 + len(block))
    return text.replace("```python", "").replace("```py", "").replace("```", "").strip()


def extract_code(raw_output: str, entry_point: str) -> str:
    text = strip_markdown_fences(raw_output)
    for marker in ("Explanation:", "Example:", "Examples:", "# Explanation"):
        marker_index = text.find(marker)
        if marker_index != -1:
            text = text[:marker_index].strip()
    match = re.search(rf"def\s+{re.escape(entry_point)}\s*\(", text)
    if match:
        imports = [
            line.strip()
            for line in text[: match.start()].splitlines()
            if line.strip().startswith(("import ", "from "))
        ]
        function_code = text[match.start() :].strip()
        return "\n".join(imports + ([""] if imports else []) + [function_code]).strip()
    return text.strip()


def evaluate(task: Task, raw_output: str, timeout_s: int) -> tuple[bool, str | None]:
    code = extract_code(raw_output, task.entry_point)
    try:
        ast.parse(code)
    except SyntaxError as error:
        return False, f"SyntaxError: {error.msg} at line {error.lineno}"

    prelude = (
        "from typing import *\nimport math\nimport re\nimport itertools\n"
        "import collections\nimport functools\nimport heapq\nimport bisect\n"
        "import string\nimport statistics\nfrom collections import *\n\n"
    )
    source = prelude + code + "\n\n" + task.test + f"\n\ncheck({task.entry_point})\n"
    with tempfile.TemporaryDirectory() as temp_dir:
        candidate_path = Path(temp_dir) / "candidate.py"
        candidate_path.write_text(source, encoding="utf-8")
        try:
            result = subprocess.run(
                [sys.executable, str(candidate_path)],
                cwd=temp_dir,
                capture_output=True,
                text=True,
                timeout=timeout_s,
            )
        except subprocess.TimeoutExpired:
            return False, f"Timeout: exceeded {timeout_s}s"
    if result.returncode == 0:
        return True, None
    stderr = (result.stderr or result.stdout or "unknown execution failure").strip()
    return False, stderr[-800:]


def model_input_device(model: Any) -> torch.device:
    return model.get_input_embeddings().weight.device


def entropy_and_top2(logits: torch.Tensor, tokenizer: Any, position: int) -> TokenTrace:
    logits = logits.float()
    log_probabilities = torch.log_softmax(logits, dim=-1)
    probabilities = log_probabilities.exp()
    entropy = float((-(probabilities * log_probabilities).sum()).item())
    top_values, top_ids = torch.topk(logits, k=2, dim=-1)
    top1_id, top2_id = int(top_ids[0].item()), int(top_ids[1].item())
    return TokenTrace(
        position=position,
        token_id=top1_id,
        token_text=tokenizer.decode([top1_id]),
        entropy=entropy,
        margin=float((top_values[0] - top_values[1]).item()),
        top1_id=top1_id,
        top1_text=tokenizer.decode([top1_id]),
        top2_id=top2_id,
        top2_text=tokenizer.decode([top2_id]),
    )


@torch.inference_mode()
def greedy_generate(model: Any, tokenizer: Any, prompt: str, max_new_tokens: int) -> Generation:
    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    started = time.perf_counter()
    outputs = model(prompt_ids, use_cache=True)
    cache = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]
    generated_ids: list[int] = []
    trace: list[TokenTrace] = []
    eos_token_id = tokenizer.eos_token_id

    for position in range(max_new_tokens):
        token_trace = entropy_and_top2(next_logits[0], tokenizer, position)
        trace.append(token_trace)
        next_token_id = token_trace.top1_id
        generated_ids.append(next_token_id)
        if eos_token_id is not None and next_token_id == eos_token_id:
            break
        next_token = torch.tensor([[next_token_id]], device=input_device, dtype=torch.long)
        outputs = model(next_token, past_key_values=cache, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :]

    return Generation(
        token_ids=generated_ids,
        code=tokenizer.decode(generated_ids, skip_special_tokens=True),
        trace=trace,
        latency_s=time.perf_counter() - started,
    )


@torch.inference_mode()
def generate_top1_top2_batch(
    model: Any,
    tokenizer: Any,
    prompt: str,
    prefix_ids: list[int],
    top1_id: int,
    top2_id: int,
    max_new_tokens: int,
) -> tuple[Generation, Generation]:
    """Continue top-1 and top-2 branches in a batch of two equally long prefixes."""

    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    prefix_tensor = torch.tensor(prefix_ids, dtype=torch.long, device=input_device).unsqueeze(0)
    shared_prefix = torch.cat([prompt_ids, prefix_tensor], dim=1) if prefix_ids else prompt_ids
    candidates = torch.tensor([[top1_id], [top2_id]], dtype=torch.long, device=input_device)
    input_ids = torch.cat([shared_prefix.repeat(2, 1), candidates], dim=1)
    started = time.perf_counter()
    outputs = model(input_ids, use_cache=True)
    cache = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]
    branch_ids = [[top1_id], [top2_id]]
    branch_trace: list[list[TokenTrace]] = [[], []]
    finished = [False, False]
    eos_token_id = tokenizer.eos_token_id
    remaining = max(max_new_tokens - len(prefix_ids) - 1, 0)

    for step in range(remaining):
        next_ids: list[int] = []
        for branch_index in range(2):
            token_trace = entropy_and_top2(next_logits[branch_index], tokenizer, len(prefix_ids) + 1 + step)
            branch_trace[branch_index].append(token_trace)
            token_id = eos_token_id if finished[branch_index] and eos_token_id is not None else token_trace.top1_id
            branch_ids[branch_index].append(int(token_id))
            if eos_token_id is not None and token_id == eos_token_id:
                finished[branch_index] = True
            next_ids.append(int(token_id))
        if all(finished):
            break
        next_tensor = torch.tensor(next_ids, dtype=torch.long, device=input_device).unsqueeze(1)
        outputs = model(next_tensor, past_key_values=cache, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :]

    latency_s = time.perf_counter() - started
    complete_ids = [prefix_ids + branch for branch in branch_ids]
    return (
        Generation(
            token_ids=complete_ids[0],
            code=tokenizer.decode(complete_ids[0], skip_special_tokens=True),
            trace=branch_trace[0],
            latency_s=latency_s,
        ),
        Generation(
            token_ids=complete_ids[1],
            code=tokenizer.decode(complete_ids[1], skip_special_tokens=True),
            trace=branch_trace[1],
            latency_s=latency_s,
        ),
    )


def select_positions(
    trace: list[TokenTrace],
    candidates_per_task: int,
    controls_per_task: int,
    edge_buffer: int,
    rng: np.random.Generator,
) -> list[tuple[str, TokenTrace]]:
    valid = trace[edge_buffer : max(len(trace) - edge_buffer, edge_buffer)]
    high_entropy = sorted(valid, key=lambda item: item.entropy, reverse=True)[:candidates_per_task]
    high_positions = {item.position for item in high_entropy}
    controls_pool = [item for item in valid if item.position not in high_positions]
    controls_count = min(controls_per_task, len(controls_pool))
    controls = (
        [controls_pool[index] for index in rng.choice(len(controls_pool), size=controls_count, replace=False)]
        if controls_count
        else []
    )
    return [("high_entropy", item) for item in high_entropy] + [("random_control", item) for item in controls]


def baseline_record(task: Task, generation: Generation, passed: bool, error: str | None) -> dict[str, Any]:
    return {
        "task_id": task.task_id,
        "passed": passed,
        "error": error,
        "code": generation.code,
        "token_ids": generation.token_ids,
        "trace": [asdict(item) for item in generation.trace],
        "generation_latency_s": generation.latency_s,
    }


def write_summary(branch_records: list[dict[str, Any]], output_dir: Path) -> None:
    rows = []
    for selection_type in ("high_entropy", "random_control"):
        group = [record for record in branch_records if record["selection_type"] == selection_type]
        if not group:
            continue
        recoverable = sum(bool(record["recoverable"]) for record in group)
        rows.append(
            {
                "selection_type": selection_type,
                "n_positions": len(group),
                "recoverable": recoverable,
                "recovery_rate": recoverable / len(group),
                "mean_entropy": float(np.mean([record["entropy"] for record in group])),
                "mean_extra_tokens": float(np.mean([record["extra_tokens"] for record in group])),
            }
        )
    report = {"rows": rows, "created_at_unix": time.time()}
    (output_dir / "summary.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    markdown = ["# Top-1 vs Top-2 counterfactual pilot", "", "| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |", "|---|---:|---:|---:|---:|---:|"]
    markdown.extend(
        "| {selection_type} | {n_positions} | {recoverable} | {recovery_rate:.1%} | {mean_entropy:.3f} | {mean_extra_tokens:.1f} |".format(**row)
        for row in rows
    )
    (output_dir / "summary.md").write_text("\n".join(markdown) + "\n", encoding="utf-8")
    print("\n".join(markdown))


def main() -> None:
    args = parse_args()
def run_notebook(
    *,
    task_ids: str,
    model_name: str = DEFAULT_MODEL,
    output_dir: str = "/kaggle/working/top2_counterfactual_pilot",
    max_new_tokens: int = 256,
    candidates_per_task: int = 3,
    controls_per_task: int = 3,
    edge_buffer: int = 5,
    timeout_s: int = 8,
    seed: int = 42,
    trust_remote_code: bool = False,
    overwrite: bool = False,
) -> None:
    """Run the pilot directly from a Kaggle notebook cell.

    Example:
        run_notebook(task_ids="HumanEval/26,HumanEval/38", candidates_per_task=2)
    """

    args = argparse.Namespace(
        model_name=model_name,
        output_dir=output_dir,
        task_ids=task_ids,
        num_tasks=10,
        max_new_tokens=max_new_tokens,
        candidates_per_task=candidates_per_task,
        controls_per_task=controls_per_task,
        edge_buffer=edge_buffer,
        timeout_s=timeout_s,
        seed=seed,
        trust_remote_code=trust_remote_code,
        overwrite=overwrite,
    )
    main(args)


def main(args: argparse.Namespace | None = None) -> None:
    args = args or parse_args()
    if not torch.cuda.is_available():
        raise RuntimeError("This pilot requires a Kaggle GPU session.")
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    rng = np.random.default_rng(args.seed)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    baseline_path = output_dir / "baselines.jsonl"
    branch_path = output_dir / "branches.jsonl"
    metadata_path = output_dir / "metadata.json"
    if args.overwrite:
        for path in (baseline_path, branch_path, metadata_path):
            if path.exists():
                path.unlink()

    tasks = load_tasks(args.task_ids, args.num_tasks)
    metadata_path.write_text(
        json.dumps({"args": vars(args), "tasks": [task.task_id for task in tasks]}, indent=2),
        encoding="utf-8",
    )
    print(f"Loading one model across {torch.cuda.device_count()} GPU(s): {args.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=args.trust_remote_code)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=args.trust_remote_code,
    )
    model.eval()

    existing_baselines = {record["task_id"]: record for record in read_jsonl(baseline_path)}
    existing_branches = {record["task_id"] for record in read_jsonl(branch_path)}
    for task in tasks:
        if task.task_id not in existing_baselines:
            print(f"Baseline {task.task_id}")
            prompt = build_prompt(task, tokenizer)
            generation = greedy_generate(model, tokenizer, prompt, args.max_new_tokens)
            passed, error = evaluate(task, generation.code, args.timeout_s)
            record = baseline_record(task, generation, passed, error)
            append_jsonl(baseline_path, record)
            existing_baselines[task.task_id] = record
            print(f"  {'PASS' if passed else 'FAIL'} | {len(generation.token_ids)} tokens")

        baseline = existing_baselines[task.task_id]
        if baseline["passed"]:
            print(f"Skip {task.task_id}: baseline passed (pilot targets failures).")
            continue
        if task.task_id in existing_branches:
            print(f"Skip {task.task_id}: branches already saved.")
            continue

        prompt = build_prompt(task, tokenizer)
        trace = [TokenTrace(**item) for item in baseline["trace"]]
        selected = select_positions(
            trace,
            candidates_per_task=args.candidates_per_task,
            controls_per_task=args.controls_per_task,
            edge_buffer=args.edge_buffer,
            rng=rng,
        )
        if not selected:
            print(f"Skip {task.task_id}: too few generated tokens for valid branch positions.")
            continue
        print(f"Branching {task.task_id}: {len(selected)} positions")
        for selection_type, point in selected:
            prefix_ids = baseline["token_ids"][: point.position]
            top1_generation, top2_generation = generate_top1_top2_batch(
                model,
                tokenizer,
                prompt,
                prefix_ids=prefix_ids,
                top1_id=point.top1_id,
                top2_id=point.top2_id,
                max_new_tokens=args.max_new_tokens,
            )
            top1_passed, top1_error = evaluate(task, top1_generation.code, args.timeout_s)
            top2_passed, top2_error = evaluate(task, top2_generation.code, args.timeout_s)
            record = {
                "task_id": task.task_id,
                "selection_type": selection_type,
                "position": point.position,
                "position_relative": point.position / max(len(baseline["token_ids"]) - 1, 1),
                "entropy": point.entropy,
                "margin": point.margin,
                "top1_token": point.top1_text,
                "top2_token": point.top2_text,
                "top1_passed": top1_passed,
                "top1_error": top1_error,
                "top2_passed": top2_passed,
                "top2_error": top2_error,
                "recoverable": (not top1_passed) and top2_passed,
                "top1_matches_baseline": top1_generation.code == baseline["code"],
                "extra_tokens": len(top1_generation.token_ids) + len(top2_generation.token_ids) - 2 * len(prefix_ids),
                "branch_latency_s": top1_generation.latency_s,
                "top1_code": top1_generation.code,
                "top2_code": top2_generation.code,
            }
            append_jsonl(branch_path, record)
            print(
                f"  {selection_type} t={point.position:>3} H={point.entropy:.3f} "
                f"top1={'P' if top1_passed else 'F'} top2={'P' if top2_passed else 'F'}"
            )
        existing_branches.add(task.task_id)
        write_summary(read_jsonl(branch_path), output_dir)

    write_summary(read_jsonl(branch_path), output_dir)
    print(f"\nSaved resumable outputs to: {output_dir}")


if __name__ == "__main__":
    main()

README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loading one model across 2 GPU(s): Qwen/Qwen2.5-Coder-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Skip HumanEval/0: baseline passed (pilot targets failures).
Skip HumanEval/1: baseline passed (pilot targets failures).
Skip HumanEval/2: baseline passed (pilot targets failures).
Skip HumanEval/3: baseline passed (pilot targets failures).
Skip HumanEval/4: baseline passed (pilot targets failures).
Skip HumanEval/5: baseline passed (pilot targets failures).
Skip HumanEval/6: baseline passed (pilot targets failures).
Skip HumanEval/7: baseline passed (pilot targets failures).
Skip HumanEval/8: baseline passed (pilot targets failures).
Skip HumanEval/9: baseline passed (pilot targets failures).
# Top-1 vs Top-2 counterfactual pilot

| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |
|---|---:|---:|---:|---:|---:|
| high_entropy | 1 | 0 | 0.0% | 0.891 | 192.0 |
| random_control | 1 | 0 | 0.0% | 0.000 | 90.0 |

Saved resumable outputs to: /kaggle/working/top2_counterfactual_pilot


In [6]:
run_notebook(
    task_ids="HumanEval/26",
    candidates_per_task=1,
    controls_per_task=1,
    max_new_tokens=256,
)

Loading one model across 2 GPU(s): Qwen/Qwen2.5-Coder-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Baseline HumanEval/26
  FAIL | 53 tokens
Branching HumanEval/26: 2 positions
  high_entropy t= 18 H=0.891 top1=F top2=F
  random_control t=  8 H=0.000 top1=F top2=F
# Top-1 vs Top-2 counterfactual pilot

| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |
|---|---:|---:|---:|---:|---:|
| high_entropy | 1 | 0 | 0.0% | 0.891 | 192.0 |
| random_control | 1 | 0 | 0.0% | 0.000 | 90.0 |
# Top-1 vs Top-2 counterfactual pilot

| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |
|---|---:|---:|---:|---:|---:|
| high_entropy | 1 | 0 | 0.0% | 0.891 | 192.0 |
| random_control | 1 | 0 | 0.0% | 0.000 | 90.0 |

Saved resumable outputs to: /kaggle/working/top2_counterfactual_pilot


In [2]:
"""
This pilot enumerates every eligible token position of HumanEval/26, forces
the model's top-2 token, completes greedily, evaluates the result, and saves
resumable JSONL/CSV outputs.
"""

import ast
import csv
import gc
import json
import keyword
import shutil
import time
from collections import Counter
from dataclasses import asdict
from pathlib import Path
from typing import Any

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


PILOT_TASK_ID = "HumanEval/26"
PILOT_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
PILOT_OUTPUT_DIR = Path("/kaggle/working/exhaustive_branch_humaneval_26_qwen25_7b")
PILOT_MAX_NEW_TOKENS = 256
PILOT_EDGE_BUFFER = 2
PILOT_TIMEOUT_S = 8


def classify_token(token_text: str) -> str:
    """Coarse lexical category used only for post-hoc interpretation."""
    if token_text == "":
        return "empty"
    if "\n" in token_text:
        return "newline"
    if token_text.isspace():
        return "whitespace"
    stripped = token_text.strip()
    if keyword.iskeyword(stripped):
        return "keyword"
    if stripped.isidentifier():
        return "identifier"
    if stripped in {
        "+", "-", "*", "/", "//", "%", "**", "=", "==", "!=", "<", ">",
        "<=", ">=", ":=", "&", "|", "^", "~", "<<", ">>",
    }:
        return "operator"
    try:
        ast.literal_eval(stripped)
        return "literal"
    except Exception:
        return "other"


def code_ast_signature(raw_output: str, entry_point: str) -> dict[str, Any]:
    """Return a small AST signature without treating it as a selection signal."""
    code = extract_code(raw_output, entry_point)
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return {
            "parses": False,
            "node_count": 0,
            "node_types": [],
        }
    node_types = [type(node).__name__ for node in ast.walk(tree)]
    return {
        "parses": True,
        "node_count": len(node_types),
        "node_types": sorted(set(node_types)),
    }


def ast_distances(
    baseline_signature: dict[str, Any],
    branch_signature: dict[str, Any],
) -> tuple[float, float]:
    """Jaccard type distance and normalized AST-size distance."""
    baseline_types = set(baseline_signature["node_types"])
    branch_types = set(branch_signature["node_types"])
    union = baseline_types | branch_types
    type_distance = (
        1.0 - len(baseline_types & branch_types) / len(union)
        if union
        else 0.0
    )
    baseline_size = baseline_signature["node_count"]
    branch_size = branch_signature["node_count"]
    size_distance = abs(baseline_size - branch_size) / max(
        baseline_size,
        branch_size,
        1,
    )
    return type_distance, size_distance


@torch.inference_mode()
def generate_forced_token_branch(
    model: Any,
    tokenizer: Any,
    prompt: str,
    prefix_ids: list[int],
    forced_token_id: int,
    max_new_tokens: int,
) -> Generation:
    """Force one token after a greedy prefix, then continue greedily.

    Only the alternative branch is generated. The corresponding top-1 branch
    is the already-saved deterministic greedy baseline.
    """
    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    generated_ids = list(prefix_ids) + [int(forced_token_id)]
    generated_tensor = torch.tensor(
        [generated_ids],
        dtype=torch.long,
        device=input_device,
    )
    input_ids = torch.cat([prompt_ids, generated_tensor], dim=1)
    eos_token_id = tokenizer.eos_token_id

    started = time.perf_counter()
    if eos_token_id is None or forced_token_id != eos_token_id:
        outputs = model(input_ids, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :]
        remaining = max(max_new_tokens - len(generated_ids), 0)

        for _ in range(remaining):
            next_token_id = int(torch.argmax(next_logits[0]).item())
            generated_ids.append(next_token_id)
            if eos_token_id is not None and next_token_id == eos_token_id:
                break
            next_token = torch.tensor(
                [[next_token_id]],
                dtype=torch.long,
                device=input_device,
            )
            outputs = model(next_token, past_key_values=cache, use_cache=True)
            cache = outputs.past_key_values
            next_logits = outputs.logits[:, -1, :]

    return Generation(
        token_ids=generated_ids,
        code=tokenizer.decode(generated_ids, skip_special_tokens=True),
        trace=[],
        latency_s=time.perf_counter() - started,
    )


def write_exhaustive_summary(
    records: list[dict[str, Any]],
    output_dir: Path,
    verbose: bool = False,
) -> None:
    records = sorted(records, key=lambda row: row["position"])
    summary_fields = [
        "position",
        "position_relative",
        "entropy",
        "margin",
        "top1_share_within_top2",
        "top2_share_within_top2",
        "top1_token",
        "top2_token",
        "top1_class",
        "top2_class",
        "semantic_class_change",
        "top2_passed",
        "top2_parses",
        "ast_type_distance",
        "ast_size_distance",
        "branch_tokens",
        "branch_latency_s",
        "error",
    ]
    with (output_dir / "exhaustive_summary.csv").open(
        "w",
        encoding="utf-8",
        newline="",
    ) as handle:
        writer = csv.DictWriter(handle, fieldnames=summary_fields)
        writer.writeheader()
        for record in records:
            writer.writerow({field: record.get(field) for field in summary_fields})

    recoveries = [record for record in records if record["top2_passed"]]
    by_entropy = sorted(records, key=lambda row: row["entropy"], reverse=True)
    entropy_rank = {
        record["position"]: rank
        for rank, record in enumerate(by_entropy, start=1)
    }
    report = {
        "task_id": PILOT_TASK_ID,
        "model": PILOT_MODEL,
        "evaluated_positions": len(records),
        "recoveries": len(recoveries),
        "recovery_positions": [record["position"] for record in recoveries],
        "recovery_entropy_ranks": [
            entropy_rank[record["position"]] for record in recoveries
        ],
        "highest_entropy_positions": [
            {
                "position": record["position"],
                "entropy": record["entropy"],
                "top1_token": record["top1_token"],
                "top2_token": record["top2_token"],
                "top2_passed": record["top2_passed"],
            }
            for record in by_entropy[:10]
        ],
    }
    (output_dir / "decision_report.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    if verbose:
        print("\n=== DECISION REPORT ===")
        print(json.dumps(report, indent=2, ensure_ascii=False))


def run_exhaustive_pilot(overwrite: bool = False) -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a GPU accelerator in Kaggle before running.")

    torch.manual_seed(42)
    PILOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    baseline_path = PILOT_OUTPUT_DIR / "baseline.json"
    branches_path = PILOT_OUTPUT_DIR / "exhaustive_branches.jsonl"

    if overwrite:
        for path in (
            baseline_path,
            branches_path,
            PILOT_OUTPUT_DIR / "exhaustive_summary.csv",
            PILOT_OUTPUT_DIR / "decision_report.json",
        ):
            if path.exists():
                path.unlink()

    tasks = load_tasks(PILOT_TASK_ID, num_tasks=1)
    if len(tasks) != 1 or tasks[0].task_id != PILOT_TASK_ID:
        raise RuntimeError(f"Could not load exactly {PILOT_TASK_ID}.")
    task = tasks[0]

    gc.collect()
    torch.cuda.empty_cache()
    print(f"Loading {PILOT_MODEL} on {torch.cuda.device_count()} GPU(s)...")
    tokenizer = AutoTokenizer.from_pretrained(PILOT_MODEL)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        PILOT_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()
    prompt = build_prompt(task, tokenizer)

    if baseline_path.exists():
        baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
        print("Loaded saved baseline.")
    else:
        print(f"Generating greedy baseline for {PILOT_TASK_ID}...")
        baseline_generation = greedy_generate(
            model,
            tokenizer,
            prompt,
            PILOT_MAX_NEW_TOKENS,
        )
        baseline_passed, baseline_error = evaluate(
            task,
            baseline_generation.code,
            PILOT_TIMEOUT_S,
        )
        baseline = baseline_record(
            task,
            baseline_generation,
            baseline_passed,
            baseline_error,
        )
        baseline_path.write_text(
            json.dumps(baseline, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

    if baseline["passed"]:
        raise RuntimeError(
            "The greedy baseline passed, so recoverability is undefined for "
            "this run. Stop and send the baseline.json file back."
        )

    trace = [TokenTrace(**item) for item in baseline["trace"]]
    baseline_signature = code_ast_signature(baseline["code"], task.entry_point)
    special_ids = set(tokenizer.all_special_ids)
    eligible_points = [
        point
        for point in trace[
            PILOT_EDGE_BUFFER : max(
                len(trace) - PILOT_EDGE_BUFFER,
                PILOT_EDGE_BUFFER,
            )
        ]
        if point.top1_id not in special_ids and point.top2_id not in special_ids
    ]
    print(
        f"Baseline FAIL with {len(baseline['token_ids'])} tokens. "
        f"Eligible positions: {len(eligible_points)}"
    )

    existing_records = read_jsonl(branches_path)
    completed_positions = {record["position"] for record in existing_records}

    for index, point in enumerate(eligible_points, start=1):
        if point.position in completed_positions:
            continue

        prefix_ids = baseline["token_ids"][: point.position]
        branch = generate_forced_token_branch(
            model,
            tokenizer,
            prompt,
            prefix_ids,
            point.top2_id,
            PILOT_MAX_NEW_TOKENS,
        )
        passed, error = evaluate(task, branch.code, PILOT_TIMEOUT_S)
        branch_signature = code_ast_signature(branch.code, task.entry_point)
        ast_type_distance, ast_size_distance = ast_distances(
            baseline_signature,
            branch_signature,
        )
        probability_ratio = float(torch.exp(torch.tensor(-point.margin)).item())
        top1_share_within_top2 = 1.0 / (1.0 + probability_ratio)
        top2_share_within_top2 = probability_ratio / (1.0 + probability_ratio)
        top1_class = classify_token(point.top1_text)
        top2_class = classify_token(point.top2_text)

        record = {
            "task_id": task.task_id,
            "model": PILOT_MODEL,
            "position": point.position,
            "position_relative": point.position
            / max(len(baseline["token_ids"]) - 1, 1),
            "entropy": point.entropy,
            "margin": point.margin,
            # Shares renormalized over top-1 and top-2 only, not full-vocab probabilities.
            "top1_share_within_top2": top1_share_within_top2,
            "top2_share_within_top2": top2_share_within_top2,
            "top1_token_id": point.top1_id,
            "top2_token_id": point.top2_id,
            "top1_token": point.top1_text,
            "top2_token": point.top2_text,
            "top1_class": top1_class,
            "top2_class": top2_class,
            "semantic_class_change": top1_class != top2_class,
            "top2_passed": passed,
            "top2_parses": branch_signature["parses"],
            "ast_type_distance": ast_type_distance,
            "ast_size_distance": ast_size_distance,
            "branch_tokens": len(branch.token_ids),
            "branch_latency_s": branch.latency_s,
            "error": error,
            "top2_code": branch.code,
        }
        append_jsonl(branches_path, record)
        existing_records.append(record)
        completed_positions.add(point.position)
        print(
            f"[{index:>3}/{len(eligible_points)}] "
            f"t={point.position:>3} H={point.entropy:>6.3f} "
            f"{point.top1_text!r} -> {point.top2_text!r} "
            f"{'PASS' if passed else 'FAIL'}"
        )

        # Keep the summary usable even if Kaggle interrupts the session.
        write_exhaustive_summary(existing_records, PILOT_OUTPUT_DIR, verbose=False)

    write_exhaustive_summary(existing_records, PILOT_OUTPUT_DIR, verbose=True)
    archive_path = shutil.make_archive(
        str(PILOT_OUTPUT_DIR),
        "zip",
        root_dir=PILOT_OUTPUT_DIR,
    )
    print("\nDownload this ZIP from Kaggle Output and send it back:")
    print(archive_path)


# First run: preserves and resumes partial output if Kaggle disconnects.
run_exhaustive_pilot(overwrite=False)


README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loading Qwen/Qwen2.5-Coder-7B-Instruct on 2 GPU(s)...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generating greedy baseline for HumanEval/26...
Baseline FAIL with 53 tokens. Eligible positions: 49
[  1/49] t=  2 H= 0.000 ' import' -> ' List' FAIL
[  2/49] t=  3 H= 0.422 ' List' -> ' *\n' FAIL
[  3/49] t=  4 H= 0.200 '\n\n\n' -> '\n' PASS
[  4/49] t=  5 H= 0.002 'def' -> 'from' PASS
[  5/49] t=  6 H= 0.000 ' remove' -> 'remove' FAIL
[  6/49] t=  7 H= 0.000 '_duplicates' -> ' duplicates' FAIL
[  7/49] t=  8 H= 0.000 '(numbers' -> '(nums' FAIL
[  8/49] t=  9 H= 0.000 ':' -> ':\n' FAIL
[  9/49] t= 10 H= 0.000 ' List' -> 'List' FAIL
[ 10/49] t= 11 H= 0.000 '[int' -> '(int' FAIL
[ 11/49] t= 12 H= 0.000 '])' -> ']' FAIL
[ 12/49] t= 13 H= 0.000 ' ->' -> ' -' FAIL
[ 13/49] t= 14 H= 0.000 ' List' -> 'List' FAIL
[ 14/49] t= 15 H= 0.000 '[int' -> '(int' FAIL
[ 15/49] t= 16 H= 0.002 ']:\n' -> ']:' FAIL
[ 16/49] t= 17 H= 0.002 '   ' -> '    \n' FAIL
[ 17/49] t= 18 H= 0.891 ' seen' -> ' """' FAIL
[ 18/49] t= 19 H= 0.008 ' =' -> '_once' FAIL
[ 19/49] t= 20 H= 0.050 ' set' -> ' {}\n' FAIL
[ 20/49]